In [1]:
from google.colab import drive
import torch
import sys
import numpy as np
import math
import time
import torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup
drive.mount('/content/gdrive', force_remount=True)


base_dir = "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2"
# base_dir = "/content/gdrive/MyDrive/Final Project"
sys.path.append(base_dir)

device = torch.device("cuda" if torch.cuda.is_available()
                else ("mps" if torch.backends.mps.is_available()
                else "cpu"))
#device = torch.device("meta")
print(F"Device set to {device}")


Mounted at /content/gdrive
Device set to cuda


In [2]:
import importlib
import Models.GPT_Model as GPT_Model
import Datasets.DataLoader as DataLoader_Lib

importlib.reload(GPT_Model)
importlib.reload(DataLoader_Lib)

from Models.GPT_Model import GPT2_Lag, GPTConfig
from Models.BERT_Model import BERT_Lag, BERTConfig
from Datasets.DataLoader import TinyShakespeareDataLoader, TinyStoriesDataLoader, CombinedBinDataLoader

In [3]:
import gc
try:
    del model, optimizer, scheduler, train_loader, val_loader
except: pass

torch.cuda.empty_cache()
gc.collect()

269

In [4]:
batch_per_iter = 32 # Adjust batch size based on your Colab GPU memory
grad_acc_factor = 8
eff_batch_size = batch_per_iter * grad_acc_factor
block_size = 512
warmup_steps = 300
num_steps_train = 3000
num_steps_val = 10
weight_decay = .1
dropout = 0.
lr = 1e-4
tokens_per_step = eff_batch_size * block_size

config = BERTConfig(num_heads = 12,
  num_layers = 12,
  vocab_size = 50257,
  embedding_dim = 768,
  block_size = block_size,
  dropout = dropout,
  weight_decay = weight_decay,
  pad_token_id=50256)

model = BERT_Lag(config, device)
model = model.to(device)
model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_steps_train
)
scaler = torch.amp.GradScaler('cuda')

In [5]:
!cp "/content/gdrive/MyDrive/Junior/Second Semester/CS 6787 - Advanced ML Systems/regen-gpt2/Datasets/fineweb_1B.bin" "/content/fineweb_1B.bin"
train_loader, val_loader = CombinedBinDataLoader.create_loaders(
    '/content/fineweb_1B.bin', batch_per_iter, block_size, config, seed=0
)

Initialized loader with 54,931 chunks of size 16385.
Initialized loader with 6,104 chunks of size 16385.


In [6]:
@torch.no_grad()
def estimate_loss(model, loader, device, eval_iters=10, lam=.5):
    model.eval()

    losses_fwd = []
    losses_bwd = []

    for i in range(eval_iters):
        x, y, _ = loader.get_data()
        x, y = x.to(device), y.to(device)
        use_fwd = i % 2 == 0

        loss = model(x, y, use_fwd)
        if(use_fwd):
          losses_fwd.append(loss.item())
        else:
          losses_bwd.append(loss.item())


    model.train()

    avg_fwd = sum(losses_fwd) / len(losses_fwd)
    avg_bwd = sum(losses_bwd) / len(losses_bwd)

    ppl_fwd = torch.exp(torch.tensor(avg_fwd)).item()
    ppl_bwd = torch.exp(torch.tensor(avg_bwd)).item()

    return avg_fwd, avg_bwd, ppl_fwd, ppl_bwd



def train_loop(model, optimizer, scheduler, device, train_loader, val_loader,
               num_steps_train, num_steps_val):

    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    loss_fwd, loss_bwd, ppl_fwd, ppl_bwd = estimate_loss(model, val_loader, device, num_steps_val)
    print(f"Step    0 | Val FWD: {loss_fwd:.4f} | PPL_FWD: {ppl_fwd:.2f} | Val BWD: {loss_bwd:.4f}| PPL_BWD: {ppl_bwd:.2f}")
    start = time.time()
    tokens_seen = 0

    #Define some useful constants

    for step in range(num_steps_train):
        if step % 50 == 0 and step > 0:
            loss_fwd, loss_bwd, ppl_fwd, ppl_bwd = estimate_loss(model, val_loader, device, num_steps_val)
            print(f"Step  {step}| Val FWD: {loss_fwd:.4f} | PPL_FWD: {ppl_fwd:.2f} | Val BWD: {loss_bwd:.4f}| PPL_BWD: {ppl_bwd:.2f}")
            if device == 'mps':
                torch.mps.empty_cache()

        avg_loss = 0
        optimizer.zero_grad(set_to_none=True)
        for i in range(grad_acc_factor):
            x, y, _ = train_loader.get_data()
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)


            with torch.autocast(device_type='cuda', dtype=torch.float16):
                loss = model(x, y, step % 2 == 0)
                loss = loss / grad_acc_factor
                avg_loss += loss
            scaler.scale(loss).backward()

        scaler.step(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        scaler.update()
        scheduler.step()
        tokens_seen += tokens_per_step
        if step % 11 == 0:
            stop = time.time()
            if(step % 2 == 0):
              print(f"step {step:4d}/{num_steps_train} | tokens {tokens_seen:,} | Train FWD loss {avg_loss.item():.4f} | Time Since Last Train Print {(stop-start):.4f} seconds")
            else:
              print(f"step {step:4d}/{num_steps_train} | tokens {tokens_seen:,} | Train BWD loss {avg_loss.item():.4f} | Time Since Last Train Print {(stop-start):.4f} seconds")
            start = time.time()


In [ ]:
torch.cuda.empty_cache()
train_loop(model, optimizer, scheduler, device, train_loader, val_loader, num_steps_train, grad_acc_factor * 2)

Trainable parameters: 124,083,456
Step    0 | Val FWD: 10.9967 | PPL_FWD: 59676.16 | Val BWD: 10.9974| PPL_BWD: 59721.42
step    0/3000 | tokens 131,072 | Train FWD loss 10.9954 | Time Since Last Train Print 8.2758 seconds
step   11/3000 | tokens 1,572,864 | Train BWD loss 10.8538 | Time Since Last Train Print 77.7972 seconds
step   22/3000 | tokens 3,014,656 | Train FWD loss 10.4529 | Time Since Last Train Print 75.2937 seconds
step   33/3000 | tokens 4,456,448 | Train BWD loss 9.9875 | Time Since Last Train Print 72.6002 seconds
step   44/3000 | tokens 5,898,240 | Train FWD loss 9.6312 | Time Since Last Train Print 71.5597 seconds
Step  50| Val FWD: 9.5233 | PPL_FWD: 13675.13 | Val BWD: 9.5478| PPL_BWD: 14014.34
step   55/3000 | tokens 7,340,032 | Train BWD loss 9.4375 | Time Since Last Train Print 91.4935 seconds
step   66/3000 | tokens 8,781,824 | Train FWD loss 9.2065 | Time Since Last Train Print 71.3526 seconds
step   77/3000 | tokens 10,223,616 | Train BWD loss 9.0095 | Time Si

In [ ]:
loss_fwd, ppl, loss_bwd = estimate_loss(model, val_loader, device, grad_acc_factor)
print(f"Val FWD: {loss_fwd:.4f} | PPL: {ppl:.2f} | Val BWD: {-100:.4f}")